In [19]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

CLIENT_ID = os.getenv("DB_CLIENT_ID")
API_KEY = os.getenv("DB_API_KEY")

print("Client ID:", CLIENT_ID[:8] + "..." if CLIENT_ID else "MISSING")
print("API Key:", "loaded" if API_KEY else "MISSING")

Client ID: bb989c0e...
API Key: loaded


In [20]:
import requests

BASE_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/timetables/v1"

headers = {
    "DB-Client-Id": CLIENT_ID,
    "DB-Api-Key": API_KEY,
    "Accept": "application/xml",
}

# Step 1: find Berlin Hbf's eva number
response = requests.get(
    f"{BASE_URL}/station/BLS",   # BLS = official DS100 code for Berlin Hbf
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("First 500 chars:")
print(response.text[:500])

Status: 200
Content-Type: application/xml;charset=UTF-8
First 500 chars:
<stations>

<station p="11|12 D - G|12|13 A - D|13|14|13 D - G|13 A - C|13 C - D|11 D - G|14 A - D|14 A - C|14 C - D|14 E - F|11 C - D|13 E - F|11 E - F|11 A - D|12 A - D|14 D - G|12 C - D|12 E - F" meta="8070952|8089021|8098160" name="Berlin Hbf" eva="8011160" ds100="BLS" db="true" creationts="26-05-08 10:15:24.735"/>

</stations>



In [21]:
from datetime import datetime

EVA_BERLIN_HBF = "8011160"

now = datetime.now()
date_str = now.strftime("%y%m%d")   # e.g. "260515"
hour_str = now.strftime("%H")       # e.g. "14"

print(f"Fetching planned timetable for date={date_str} hour={hour_str}")

response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Fetching planned timetable for date=260515 hour=16


Status: 200
Length: 8346 chars
First 1500 chars:
<?xml version='1.0' encoding='UTF-8'?><timetable station='Berlin Hbf'><s id="-8982296658299132323-2605151505-19"><tl f="N" t="p" o="800165" c="RE" n="3726"/><ar pt="2605151644" pp="11" l="RE7" fb="RE7" ppth="Dessau Hbf|Roßlau(Elbe)|Jeber-Bergfrieden|Medewitz(Mark)|Wiesenburg(Mark)|Bad Belzig|Baitz|Brück(Mark)|Borkheide|Beelitz-Heilstätten|Seddin|Michendorf|Wilhelmshorst|Potsdam-Rehbrücke|Potsdam Medienstadt Babelsberg|Berlin-Wannsee|Berlin-Charlottenburg|Berlin Zoologischer Garten"/><dp pt="2605151646" pp="11" l="RE7" fb="RE7" ppth="Berlin Friedrichstraße|Berlin Alexanderplatz|Berlin Ostbahnhof|Berlin Ostkreuz|Königs Wusterhausen|Zeesen|Bestensee|Groß Köris|Halbe|Oderin|Brand Tropical Islands|Schönwalde(Spreewald)|Lubolz|Lübben(Spreewald)|Lübbenau(Spreewald)|Calau(Nl)|Luckaitztal|Altdöbern|Großräschen|Sedlitz Ost|Senftenberg"/></s><s id="9074417543825272152-2605151544-9"><tl f="D" t="p" o="OWRE" c="OE" n="73775"/><ar pt="2605151635" pp="

In [22]:
response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Status: 200
Length: 111228 chars
First 1500 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="6024889151340737172-2605150616-16" eva="8011160">
    <m id="r2604793" t="h" from="2604140000" to="2605312359" cat="Information" ts="2604141735" ts-tts="26-05-14 23:16:04.764" pr="2"/>
    <m id="r2620976" t="h" from="2605150730" to="2605150930" cat="Störung" ts="2605150812" ts-tts="26-05-15 08:13:03.265" pr="1"/>
    <ar ct="2605151331"/>
    <dp ct="2605151333"/>
</s>


<s id="738303672697175305-2605151327-11" eva="8011160">
    <m id="r2621044" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605151108" ts-tts="26-05-15 11:08:21.077" pr="2"/>
    <m id="r2620992" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605150840" ts-tts="26-05-15 08:40:45.926" pr="2"/>
    <ar ct="2605151815">
        <m id="r56582449" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:36.750"/>
        <m id="r56582452" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:36

In [23]:
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime

def parse_db_time(s):
    """DB times are YYMMDDHHMM strings, e.g. '2605151644' = 2026-05-15 16:44"""
    if not s:
        return None
    return datetime.strptime(s, "%y%m%d%H%M")

def parse_plan_xml(xml_text):
    """Parse a /plan/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        tl = s.find("tl")
        ar = s.find("ar")
        dp = s.find("dp")

        row = {
            "stop_id": stop_id,
            "train_category": tl.get("c") if tl is not None else None,
            "train_number": tl.get("n") if tl is not None else None,
            "operator": tl.get("o") if tl is not None else None,
            "planned_arrival": parse_db_time(ar.get("pt")) if ar is not None else None,
            "planned_arrival_platform": ar.get("pp") if ar is not None else None,
            "arrival_line": ar.get("l") if ar is not None else None,
            "arrival_path": ar.get("ppth") if ar is not None else None,
            "planned_departure": parse_db_time(dp.get("pt")) if dp is not None else None,
            "planned_departure_platform": dp.get("pp") if dp is not None else None,
            "departure_line": dp.get("l") if dp is not None else None,
            "departure_path": dp.get("ppth") if dp is not None else None,
        }
        rows.append(row)
    return rows


# Re-fetch the plan and parse
date_str = datetime.now().strftime("%y%m%d")
hour_str = datetime.now().strftime("%H")

plan_response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

plan_rows = parse_plan_xml(plan_response.text)
plan_df = pd.DataFrame(plan_rows)

print(f"Got {len(plan_df)} scheduled stops at Berlin Hbf for hour {hour_str}")
plan_df.head(10)

Got 19 scheduled stops at Berlin Hbf for hour 16


,stop_id,train_category,train_number,operator,planned_arrival,planned_arrival_platform,arrival_line,arrival_path,planned_departure,planned_departure_platform,departure_line,departure_path
0,-8982296658299132323-2605151505-19,RE,3726,800165,2026-05-15 16:44:00,11,RE7,Dessau Hbf|Roßlau(Elbe)|Jeber-Bergfrieden|Mede...,2026-05-15 16:46:00,11,RE7,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
1,9074417543825272152-2605151544-9,OE,73775,OWRE,2026-05-15 16:35:00,11,RE1,Brandenburg Hbf|Werder(Havel)|Potsdam Park San...,2026-05-15 16:37:00,11,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
2,5526534603450682084-2605151535-6,RE,3124,800165,2026-05-15 16:24:00,12,RE2,Hennigsdorf(b Berlin)|Dallgow-Döberitz|Berlin-...,2026-05-15 16:26:00,12,RE2,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
3,5247760409549199860-2605151049-12,EC,46,51,2026-05-15 16:17:00,13,NaN,Warszawa Wschodnia|Warszawa Centralna|Warszawa...,NaT,NaN,NaN,NaN
4,-5045088658159597679-2605151556-2,ICE,140,80,2026-05-15 16:03:00,14,NaN,Berlin Ostbahnhof,2026-05-15 16:07:00,14,NaN,Berlin-Spandau|Hannover Hbf|Bünde(Westf)|Osnab...
5,-5867861389707898852-2605151538-15,OE,73740,OWRE,2026-05-15 16:59:00,13,RE1,Frankfurt(Oder)|Frankfurt(Oder)-Rosengarten|Pi...,2026-05-15 17:01:00,13,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...
6,6637172419819473914-2605151040-15,ICE,798,80,2026-05-15 16:38:00,12,NaN,Karlsruhe Hbf|Bruchsal|Heidelberg Hbf|Weinheim...,2026-05-15 16:42:00,12,NaN,Berlin Ostbahnhof
7,-359164097413752423-2605151522-7,OE,73815,OWRE,2026-05-15 16:13:00,12,RE1,Brandenburg Hbf|Werder(Havel)|Potsdam Hbf|Berl...,2026-05-15 16:15:00,12,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
8,-4743101459115365641-2605151438-15,OE,73738,OWRE,2026-05-15 15:59:00,13,RE1,Frankfurt(Oder)|Frankfurt(Oder)-Rosengarten|Pi...,2026-05-15 16:01:00,13,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...
9,-1817533299523359204-2605151507-8,OE,73780,OWRE,2026-05-15 16:22:00,13,RE1,Frankfurt(Oder)|Fürstenwalde(Spree)|Erkner|Ber...,2026-05-15 16:24:00,13,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...


In [24]:
fchg_response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", fchg_response.status_code)
print("Length:", len(fchg_response.text), "chars")
print("First 2000 chars:")
print(fchg_response.text[:2000])

Status: 200
Length: 111228 chars
First 2000 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="6024889151340737172-2605150616-16" eva="8011160">
    <m id="r2604793" t="h" from="2604140000" to="2605312359" cat="Information" ts="2604141735" ts-tts="26-05-14 23:16:04.764" pr="2"/>
    <m id="r2620976" t="h" from="2605150730" to="2605150930" cat="Störung" ts="2605150812" ts-tts="26-05-15 08:13:03.265" pr="1"/>
    <ar ct="2605151331"/>
    <dp ct="2605151333"/>
</s>


<s id="738303672697175305-2605151327-11" eva="8011160">
    <m id="r2621044" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605151108" ts-tts="26-05-15 11:08:21.077" pr="2"/>
    <m id="r2620992" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605150840" ts-tts="26-05-15 08:40:45.926" pr="2"/>
    <ar ct="2605151815">
        <m id="r56582449" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:36.750"/>
        <m id="r56582452" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:36

In [25]:
def parse_fchg_xml(xml_text):
    """Parse a /fchg/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        ar = s.find("ar")
        dp = s.find("dp")

        # collect delay reason codes
        reasons = []
        for m in s.iter("m"):
            t = m.get("t")
            c = m.get("c")
            if t == "d" and c:   # delay code
                reasons.append(c)

        row = {
            "stop_id": stop_id,
            "changed_arrival": parse_db_time(ar.get("ct")) if ar is not None and ar.get("ct") else None,
            "changed_arrival_platform": ar.get("cp") if ar is not None else None,
            "changed_departure": parse_db_time(dp.get("ct")) if dp is not None and dp.get("ct") else None,
            "changed_departure_platform": dp.get("cp") if dp is not None else None,
            "delay_codes": ",".join(sorted(set(reasons))) if reasons else None,
        }
        rows.append(row)
    return rows


fchg_rows = parse_fchg_xml(fchg_response.text)
fchg_df = pd.DataFrame(fchg_rows)

print(f"Got {len(fchg_df)} changed stops")
fchg_df.head(10)

Got 224 changed stops


,stop_id,changed_arrival,changed_arrival_platform,changed_departure,changed_departure_platform,delay_codes
0,6024889151340737172-2605150616-16,2026-05-15 13:31:00,NaN,2026-05-15 13:33:00,NaN,NaN
1,738303672697175305-2605151327-11,2026-05-15 18:15:00,NaN,2026-05-15 18:20:00,NaN,NaN
2,5253564847811370755-2605151000-13,2026-05-15 16:00:00,NaN,2026-05-15 16:04:00,NaN,24
3,9006537061621970970-2605151212-16,2026-05-15 13:57:00,NaN,2026-05-15 13:59:00,NaN,43
4,-4900972543082269967-2605151138-15,2026-05-15 13:00:00,NaN,2026-05-15 13:02:00,NaN,43
5,8777875770530232855-2605151112-16,2026-05-15 13:00:00,NaN,2026-05-15 13:02:00,NaN,43
6,-1489252697791169587-2605150825-14,2026-05-15 13:18:00,NaN,2026-05-15 13:21:00,NaN,NaN
7,-3769087447419153090-2605151252-1,NaT,NaN,2026-05-15 12:52:00,12,NaN
8,1142963549373501064-2605151128-11,2026-05-15 16:51:00,NaN,2026-05-15 16:55:00,NaN,3
9,2737274120239826370-2605151009-10,2026-05-15 14:38:00,NaN,2026-05-15 14:42:00,NaN,51


In [26]:
# Join planned with changes by stop_id
merged = plan_df.merge(fchg_df, on="stop_id", how="left")

# Compute delays in minutes
merged["arrival_delay_min"] = (
    (merged["changed_arrival"] - merged["planned_arrival"]).dt.total_seconds() / 60
)
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

# Sort by departure delay, biggest first
delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains right now")
delayed.head(20)

1 delayed trains right now


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
16,ICE,849,NaN,2026-05-15 16:20:00,2026-05-15 16:55:00,35.0,3


In [27]:
from datetime import timedelta

def fetch_plan_window(eva, hours_ahead=6):
    """Fetch /plan for the current hour + the next N hours."""
    all_rows = []
    now = datetime.now()
    for offset in range(hours_ahead + 1):
        t = now + timedelta(hours=offset)
        date_str = t.strftime("%y%m%d")
        hour_str = t.strftime("%H")
        r = requests.get(
            f"{BASE_URL}/plan/{eva}/{date_str}/{hour_str}",
            headers=headers,
            timeout=30,
        )
        if r.status_code == 200:
            all_rows.extend(parse_plan_xml(r.text))
        else:
            print(f"Hour {hour_str}: status {r.status_code}")
    return all_rows


plan_rows = fetch_plan_window(EVA_BERLIN_HBF, hours_ahead=6)
plan_df = pd.DataFrame(plan_rows)
print(f"Got {len(plan_df)} planned stops over the next 7 hours")

Got 108 planned stops over the next 7 hours


In [28]:
merged = plan_df.merge(fchg_df, on="stop_id", how="left")
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains")
delayed.head(20)

1 delayed trains


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
16,ICE,849,NaN,2026-05-15 16:20:00,2026-05-15 16:55:00,35.0,3
